In [17]:
# =========================
# 1. INSTALL + IMPORTS
# =========================
!pip install -q transformers librosa soundfile evaluate scikit-learn

import os
import random
import numpy as np
import librosa
import soundfile as sf
import torch
import evaluate

from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print("Libraries ready")

# =========================
# 2. SEED
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# 3. SETTINGS
# =========================
TARGET_SR = 16000
TARGET_DURATION = 3.0        # <-- сменяме от 5s на 3s
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)

REAL_LIMIT = 2500
AI_LIMIT = 2500

REAL_TRAIN_DIR = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/training/real"
AI_TRAIN_DIR   = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/training/fake"

REAL_TEST_DIR = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real"
AI_TEST_DIR   = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake"

NORMALIZED_DIR = "/kaggle/working/normalized_audio"
MODEL_DIR = "/kaggle/working/trained_model"

# =========================
# 4. PREPROCESSING
# =========================
def normalize_audio(input_path, output_path, target_sr=16000, target_duration=3.0):
    audio, _ = librosa.load(input_path, sr=target_sr, mono=True)
    target_length = int(target_sr * target_duration)

    if len(audio) > target_length:
        audio = audio[:target_length]
    elif len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")

    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    sf.write(output_path, audio, target_sr)

def collect_audio_files(folder):
    exts = {".wav", ".mp3", ".flac"}
    return [f for f in Path(folder).rglob("*") if f.suffix.lower() in exts]

def normalize_dataset(real_dir, fake_dir, output_dir, real_limit=250, fake_limit=250):
    output_dir = Path(output_dir)

    real_out = output_dir / "real"
    ai_out = output_dir / "ai"
    real_out.mkdir(parents=True, exist_ok=True)
    ai_out.mkdir(parents=True, exist_ok=True)

    real_files = collect_audio_files(real_dir)[:real_limit]
    fake_files = collect_audio_files(fake_dir)[:fake_limit]

    print(f"Processing REAL files: {len(real_files)}")
    skipped_real = 0
    for i, f in enumerate(real_files, 1):
        try:
            normalize_audio(str(f), str(real_out / f"{f.stem}.wav"), TARGET_SR, TARGET_DURATION)
        except Exception:
            skipped_real += 1
        if i % 50 == 0 or i == len(real_files):
            print(f"REAL: {i}/{len(real_files)}")

    print(f"Processing AI files: {len(fake_files)}")
    skipped_ai = 0
    for i, f in enumerate(fake_files, 1):
        try:
            normalize_audio(str(f), str(ai_out / f"{f.stem}.wav"), TARGET_SR, TARGET_DURATION)
        except Exception:
            skipped_ai += 1
        if i % 50 == 0 or i == len(fake_files):
            print(f"AI: {i}/{len(fake_files)}")

    print(f"Done. Skipped REAL: {skipped_real}, Skipped AI: {skipped_ai}")

normalize_dataset(
    real_dir=REAL_TRAIN_DIR,
    fake_dir=AI_TRAIN_DIR,
    output_dir=NORMALIZED_DIR,
    real_limit=REAL_LIMIT,
    fake_limit=AI_LIMIT
)

# =========================
# 5. LOAD FILES
# =========================
def load_files(data_dir):
    data_dir = Path(data_dir)
    files, labels = [], []

    for label, class_name in enumerate(["real", "ai"]):
        class_files = sorted((data_dir / class_name).glob("*.wav"))
        files.extend([str(f) for f in class_files])
        labels.extend([label] * len(class_files))
        print(f"{class_name}: {len(class_files)} files")

    return files, labels

files, labels = load_files(NORMALIZED_DIR)

print(f"Total: {len(files)}")
print(f"Real: {labels.count(0)} | AI: {labels.count(1)}")

# =========================
# 6. BETTER SPLIT
# =========================
train_files, val_files, train_labels, val_labels = train_test_split(
    files,
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

print(f"Train: {len(train_files)}")
print(f"Val:   {len(val_files)}")

# =========================
# 7. PROCESSOR + MODEL
# =========================
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,
    id2label={0: "real", 1: "ai"},
    label2id={"real": 0, "ai": 1},
    ignore_mismatched_sizes=True
)

# По-леко и по-стабилно за малък dataset
model.freeze_feature_encoder()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded on:", device)

# =========================
# 8. DATASET
# =========================
class AudioDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        audio, _ = librosa.load(self.files[idx], sr=TARGET_SR, mono=True)
        return {
            "input_values": audio,
            "labels": self.labels[idx]
        }

train_dataset = AudioDataset(train_files, train_labels)
val_dataset = AudioDataset(val_files, val_labels)

# =========================
# 9. COLLATOR
# =========================
def data_collator(features):
    audios = [f["input_values"] for f in features]
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

    batch = processor(
        audios,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )

    batch["labels"] = labels
    return batch

# =========================
# 10. METRICS
# =========================
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    precision = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# =========================
# 11. TRAINING ARGS
# =========================
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=30,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Training started...")
trainer.train()

trainer.save_model(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)

print("Training complete")

# =========================
# 12. CHUNK PREDICTION
# =========================
def preprocess_audio_array(audio, target_sr=16000, target_duration=3.0):
    target_length = int(target_sr * target_duration)

    if len(audio) > target_length:
        audio = audio[:target_length]
    elif len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")

    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    return audio.astype(np.float32)

def predict_chunk(audio_array):
    audio_array = preprocess_audio_array(audio_array, TARGET_SR, TARGET_DURATION)

    inputs = processor(
        audio_array,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    return {
        "real_prob": float(probs[0]),
        "ai_prob": float(probs[1]),
        "pred": int(np.argmax(probs))
    }

def predict_file(audio_path):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    result = predict_chunk(audio)

    label = "AI Generated" if result["pred"] == 1 else "Real Human"

    print(f"File: {audio_path}")
    print(f"Result: {label}")
    print(f"Real: {result['real_prob'] * 100:.2f}%")
    print(f"AI:   {result['ai_prob'] * 100:.2f}%")

    return result

# =========================
# 13. STREAMING SIMULATION
# =========================
def simulate_streaming(audio_path, window_sec=3.0, stride_sec=1.0, alert_threshold=0.6, trigger_count=3, history_size=5):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)

    window = int(window_sec * TARGET_SR)
    stride = int(stride_sec * TARGET_SR)

    scores = []
    alerts = []
    history = []

    for start in range(0, max(1, len(audio) - window + 1), stride):
        chunk = audio[start:start + window]
        result = predict_chunk(chunk)

        ai_prob = result["ai_prob"]
        scores.append(ai_prob)

        history.append(ai_prob > alert_threshold)
        if len(history) > history_size:
            history.pop(0)

        alert = sum(history) >= trigger_count
        alerts.append(alert)

        print(
            f"t={start/TARGET_SR:.1f}s | "
            f"AI={ai_prob*100:.2f}% | "
            f"ALERT={'YES' if alert else 'NO'}"
        )

    return scores, alerts

# Example:
# predict_file("/kaggle/input/.../somefile.wav")
# simulate_streaming("/kaggle/input/.../somefile.wav")

Libraries ready
Processing REAL files: 2500
REAL: 50/2500
REAL: 100/2500
REAL: 150/2500
REAL: 200/2500
REAL: 250/2500
REAL: 300/2500
REAL: 350/2500
REAL: 400/2500
REAL: 450/2500
REAL: 500/2500
REAL: 550/2500
REAL: 600/2500
REAL: 650/2500
REAL: 700/2500
REAL: 750/2500
REAL: 800/2500
REAL: 850/2500
REAL: 900/2500
REAL: 950/2500
REAL: 1000/2500
REAL: 1050/2500
REAL: 1100/2500
REAL: 1150/2500
REAL: 1200/2500
REAL: 1250/2500
REAL: 1300/2500
REAL: 1350/2500
REAL: 1400/2500
REAL: 1450/2500
REAL: 1500/2500
REAL: 1550/2500
REAL: 1600/2500
REAL: 1650/2500
REAL: 1700/2500
REAL: 1750/2500
REAL: 1800/2500
REAL: 1850/2500
REAL: 1900/2500
REAL: 1950/2500
REAL: 2000/2500
REAL: 2050/2500
REAL: 2100/2500
REAL: 2150/2500
REAL: 2200/2500
REAL: 2250/2500
REAL: 2300/2500
REAL: 2350/2500
REAL: 2400/2500
REAL: 2450/2500
REAL: 2500/2500
Processing AI files: 2500
AI: 50/2500
AI: 100/2500


/tmp/ipykernel_55/2756360878.py:62: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(input_path, sr=target_sr, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


AI: 150/2500
AI: 200/2500
AI: 250/2500
AI: 300/2500
AI: 350/2500
AI: 400/2500


Note: Illegal Audio-MPEG-Header 0x00000000 at offset 50.
Note: Trying to resync...
Note: Hit end of (available) data during resync.


AI: 450/2500
AI: 500/2500
AI: 550/2500
AI: 600/2500
AI: 650/2500
AI: 700/2500
AI: 750/2500
AI: 800/2500
AI: 850/2500
AI: 900/2500
AI: 950/2500
AI: 1000/2500
AI: 1050/2500
AI: 1100/2500
AI: 1150/2500
AI: 1200/2500
AI: 1250/2500
AI: 1300/2500
AI: 1350/2500
AI: 1400/2500
AI: 1450/2500
AI: 1500/2500
AI: 1550/2500
AI: 1600/2500
AI: 1650/2500
AI: 1700/2500
AI: 1750/2500
AI: 1800/2500
AI: 1850/2500
AI: 1900/2500
AI: 1950/2500
AI: 2000/2500
AI: 2050/2500
AI: 2100/2500
AI: 2150/2500
AI: 2200/2500
AI: 2250/2500
AI: 2300/2500
AI: 2350/2500


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete


In [18]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

MODEL_DIR = "/kaggle/working/trained_model"

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded successfully")
print("Device:", device)

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Model loaded successfully
Device: cuda


In [19]:
import librosa
import numpy as np
import torch

TARGET_SR = 16000
TARGET_DURATION = 3.0
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)

def preprocess_audio_array(audio, target_sr=16000, target_duration=3.0):
    target_length = int(target_sr * target_duration)

    if len(audio) > target_length:
        audio = audio[:target_length]
    elif len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")

    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    return audio.astype(np.float32)

def predict_chunk(audio_array):
    audio_array = preprocess_audio_array(audio_array, TARGET_SR, TARGET_DURATION)

    inputs = processor(
        audio_array,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    return {
        "real_prob": float(probs[0]),
        "ai_prob": float(probs[1]),
        "pred": int(np.argmax(probs))
    }

def predict_file(audio_path):
    audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    result = predict_chunk(audio)

    label = "AI Generated" if result["pred"] == 1 else "Real Human"

    print(f"File: {audio_path}")
    print(f"Result: {label}")
    print(f"Real: {result['real_prob'] * 100:.2f}%")
    print(f"AI:   {result['ai_prob'] * 100:.2f}%")

    return result

In [24]:
import os
import random
from tqdm import tqdm

def full_test(fake_path, real_path, sample_size=500, threshold=0.6):
    fake_files = [os.path.join(fake_path, f) for f in os.listdir(fake_path)
                  if f.endswith((".wav", ".mp3", ".flac"))]

    real_files = [os.path.join(real_path, f) for f in os.listdir(real_path)
                  if f.endswith((".wav", ".mp3", ".flac"))]

    fake_sample = random.sample(fake_files, min(sample_size, len(fake_files)))
    real_sample = random.sample(real_files, min(sample_size, len(real_files)))

    TP = TN = FP = FN = 0

    print("Testing AI voices...")
    for f in tqdm(fake_sample):
        result = predict_file(f)
        if result["ai_prob"] > threshold:
            TP += 1
        else:
            FN += 1

    print("Testing REAL voices...")
    for f in tqdm(real_sample):
        result = predict_file(f)
        if result["ai_prob"] <= threshold:
            TN += 1
        else:
            FP += 1

    total = TP + TN + FP + FN

    accuracy = (TP + TN) / total if total > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print("\nCONFUSION MATRIX")
    print("TP:", TP)
    print("TN:", TN)
    print("FP:", FP)
    print("FN:", FN)

    print("\nMETRICS")
    print("Accuracy :", round(accuracy * 100, 2), "%")
    print("Precision:", round(precision * 100, 2), "%")
    print("Recall   :", round(recall * 100, 2), "%")
    print("F1 score :", round(f1 * 100, 2), "%")

In [25]:
full_test(
    fake_path="/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake",
    real_path="/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real",
    sample_size=500,
    threshold=0.6
)

Testing AI voices...


  1%|          | 3/500 [00:00<00:24, 20.67it/s]

File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file2284.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file210.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file285.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file1474.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file1080.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file1632.wav
Result: Real Human
Real: 9

  2%|▏         | 10/500 [00:00<00:17, 27.90it/s]

File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file408.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file1823.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file2046.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file235.wav
Result: AI Generated
Real: 0.02%
AI:   99.98%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file188.wav
Result: AI Generated
Real: 0.03%
AI:   99.97%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/fake/file673.wav
Result: AI Generated
Real:

...

File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file1277.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file39.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file269.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file2212.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file2015.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file1121.wav
Result: Real Human
Real: 99.97%
A

 99%|█████████▉| 496/500 [00:14<00:00, 36.17it/s]

File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file605.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file38.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file1109.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file987.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file1243.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file845.wav
Result: Real Human
Real: 99.97%
AI:

100%|██████████| 500/500 [00:14<00:00, 34.66it/s]

File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file2260.wav
Result: Real Human
Real: 99.97%
AI:   0.03%
File: /kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/testing/real/file552.wav
Result: Real Human
Real: 99.97%
AI:   0.03%

CONFUSION MATRIX
TP: 367
TN: 498
FP: 2
FN: 133

METRICS
Accuracy : 86.5 %
Precision: 99.46 %
Recall   : 73.4 %
F1 score : 84.46 %


In [28]:
!zip -r /kaggle/working/final_model.zip /kaggle/working/trained_model -x "*checkpoint*"

  adding: kaggle/working/trained_model/ (stored 0%)
  adding: kaggle/working/trained_model/training_args.bin (deflated 53%)
  adding: kaggle/working/trained_model/config.json (deflated 67%)
  adding: kaggle/working/trained_model/processor_config.json (deflated 43%)
  adding: kaggle/working/trained_model/vocab.json (deflated 55%)
  adding: kaggle/working/trained_model/tokenizer_config.json (deflated 73%)
  adding: kaggle/working/trained_model/model.safetensors (deflated 9%)
